# Saving and Sharing with xarray and Zarr

The [skrf.io.xarray](../api/io/index.rst) module converts Networks, NetworkSets and Calibrations to [xarray](https://docs.xarray.dev/) Datasets, and stores them in [Zarr](https://zarr.dev/).

Unlike pickle files written with `rf.io.write`, Zarr files are safe to share, don't depend on the skrf version, and can be read by any tool that reads Zarr. Complex values are stored as complex numbers.

It needs the `xarray` extra:

```
pip install scikit-rf[xarray]
```

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

import skrf as rf
from skrf.io.xarray import add_proj, read_zarr, to_xarray, write_zarr

folder = Path(tempfile.mkdtemp())  # where this notebook writes its files

## Networks

`to_xarray` returns a Dataset, named like the Network's own properties. `s` has the dimensions `(f_hz, m, n)`, where `m` and `n` are the port indices of S<sub>mn</sub>, counting from 1. `z0` has the dimensions `(f_hz, port)`. `f_hz` is the frequency in Hz, like `Network.f`, and `f_scaled` is in the Network's display unit.

In [ ]:
ring_slot = rf.data.ring_slot
ds = to_xarray(ring_slot)
ds

## Other parameters and projections

`to_xarray` can also add other network parameters, such as `'z'`, `'y'` or `'a'` (ABCD), and projections of them and of S, such as `'db'`, `'deg'` or `'re'`. They are computed by the Network's own properties, so `s_db` here is `ring_slot.s_db`. `add_params` and `add_proj` add them to a Dataset you already have.

In [ ]:
ds = to_xarray(ring_slot, params='z', projs=['db', 'deg'])
list(ds.data_vars)

Select S21 by its port indices, and plot it against `f_scaled`:

In [ ]:
ds['s_db'].sel(m=2, n=1).plot(x='f_scaled');

`write_zarr` writes any supported object, or a Dataset made from one, with the variables added to it. `read_zarr` gives the object back:

In [ ]:
write_zarr(folder / 'ring_slot.zarr', ds)
read_zarr(folder / 'ring_slot.zarr')

## NetworkSets

A NetworkSet adds a `network` dimension. When its Networks have the same `params` keys, the params become coordinates along it, so the set can be searched by them. Here is a set of lines of three lengths and two impedances:

In [ ]:
from skrf.media import DefinedGammaZ0
from skrf.networkSet import NetworkSet

frequency = rf.Frequency(1, 10, 101, 'GHz')
lines = []
for length in (10, 20, 30):
    for impedance in (40, 50):
        media = DefinedGammaZ0(frequency, z0_port=50, z0=impedance)
        line = media.line(length, 'mm', name=f'{length} mm, {impedance} ohm')
        line.params = {'length_mm': length, 'impedance': impedance}
        lines.append(line)

ds = to_xarray(NetworkSet(lines, name='lines'))
ds

Select the 40 ohm lines by their params, and plot their return loss:

In [ ]:
lines_40 = ds.query(network='impedance == 40')
add_proj(lines_40, 'db')['s_db'].sel(m=1, n=1).plot.line(x='f_scaled', hue='length_mm');

When the params form a grid, `unstack` gives them a dimension each:

In [ ]:
grid = ds.set_index(network=['length_mm', 'impedance']).unstack('network')
grid['s'].dims

## Calibrations

A Calibration is stored as its error coefficients, one variable each, and rebuilt from them:

In [ ]:
from skrf.calibration import OnePort

media = DefinedGammaZ0(frequency)
ideals = [media.short(name='short'), media.open(name='open'), media.match(name='load')]
error = media.line(15, 'mm') ** media.attenuator(-3)
cal = OnePort([error ** ideal for ideal in ideals], ideals, name='port 1')

write_zarr(folder / 'cal.zarr', cal)
to_xarray(cal)

In [ ]:
cal = read_zarr(folder / 'cal.zarr')
dut = media.line(5, 'mm') ** media.short()
np.allclose(cal.apply_cal(error ** dut).s, dut.s)

## Limits

Objects that can't be stored yet raise `NotImplementedError`. These include the calibrations not listed in `skrf.io.xarray.CALIBRATIONS`, NetworkSets that mix Networks with and without noise, and `params` values that aren't numbers, strings or booleans.

Each file records its format version. A file written in a newer format raises an error, unless `read_zarr(..., ignore_version=True)` is used to read it anyway, at your own risk.